# 🔽 Notebook 4 — Funnel Analysis
**Measuring drop-off at every stage of the customer & order journey**

Sections:
1. Order completion funnel (Placed → Delivered)
2. Funnel by city, cuisine, payment mode
3. Conversion funnel by acquisition channel
4. Cohort retention analysis
5. RFM segmentation
6. Re-order (repeat purchase) funnel


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})

BASE = r"C:\Users\rkuma\OneDrive\Desktop\Zomato"

def load(name):
    return pd.read_csv(os.path.join(BASE, f"Zomato  Order Data.xlsx - {name}.csv"))

customers   = load("Customer")
orders      = load("Orders")
restaurants = load("Restaurants")

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"],
                                           format="%m/%d/%Y", errors="coerce")
orders["order_month"]   = orders["order_timestamp"].dt.to_period("M")
orders["order_quarter"] = orders["order_timestamp"].dt.to_period("Q")
orders["order_year"]    = orders["order_timestamp"].dt.year.astype("Int64")
orders["day_of_week"]   = orders["order_timestamp"].dt.day_name()
orders["hour"]          = orders["order_timestamp"].dt.hour
orders["discount_amount"] = orders["discount_amount"].fillna(0)
orders["delivery_fee"]    = orders["delivery_fee"].fillna(0)
orders["net_revenue"]     = orders["order_amount"] - orders["discount_amount"]
orders["is_discounted"]   = (orders["discount_amount"] > 0).astype(int)
orders["total_charge"]    = orders["net_revenue"] + orders["delivery_fee"]

customers["Signup_Time"]  = pd.to_datetime(customers["Signup_Time"],
                                           format="%d/%m/%Y", errors="coerce")
customers["signup_month"] = customers["Signup_Time"].dt.to_period("M")
customers["signup_year"]  = customers["Signup_Time"].dt.year.astype("Int64")

full = (orders
        .merge(restaurants, on="restaurant_id", how="left")
        .merge(customers,   left_on="customer_id",
               right_on="Customer_id", how="left"))

delivered  = full[full["order_status"] == "Delivered"].copy()
cancelled  = full[full["order_status"] == "Cancelled"].copy()
refunded   = full[full["order_status"] == "Refunded"].copy()

print(f"Orders: {len(orders):,} | Customers: {customers['Customer_id'].nunique():,} | Restaurants: {len(restaurants)}")
print(f"Date range: {orders['order_timestamp'].min().date()} to {orders['order_timestamp'].max().date()}")


## 1. Order Completion Funnel

In [ ]:

total_placed  = len(orders)
not_cancelled = (orders["order_status"] != "Cancelled").sum()
not_refunded  = (orders["order_status"] == "Delivered").sum()

stages = ["Orders Placed", "Not Cancelled", "Successfully Delivered"]
counts = [total_placed, not_cancelled, not_refunded]
pcts   = [c/total_placed*100 for c in counts]
drops  = [0] + [counts[i-1]-counts[i] for i in range(1, len(counts))]
drop_pcts = [0] + [(counts[i-1]-counts[i])/counts[i-1]*100 for i in range(1, len(counts))]

print("=" * 60)
print(f"{'Stage':<28} {'Count':>8}  {'% of Total':>10}  {'Drop-off':>10}")
print("=" * 60)
for s, c, p, d, dp in zip(stages, counts, pcts, drops, drop_pcts):
    drop_str = f"-{d:,} ({dp:.1f}%)" if d > 0 else "—"
    print(f"  {s:<26} {c:>8,}  {p:>9.1f}%  {drop_str:>14}")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ["#264653","#2A9D8F","#43AA8B"]
bars = ax.barh(stages[::-1], counts[::-1], color=bar_colors)
for bar, c, p in zip(bars, counts[::-1], pcts[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"{c:,}  ({p:.1f}%)", va="center", fontsize=11, fontweight="bold")
ax.set_title("Order Completion Funnel", fontsize=14, fontweight="bold")
ax.set_xlabel("Order Count")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.set_xlim(0, total_placed * 1.25)
plt.tight_layout()
plt.show()


In [ ]:

# Visualise funnel as a true funnel shape
fig, ax = plt.subplots(figsize=(8, 6))
heights = [1.0, 0.8, 0.6]
for i, (stage, count, pct, h, color) in enumerate(
    zip(stages, counts, pcts, heights,
        ["#264653","#2A9D8F","#43AA8B"])):
    rect = plt.Rectangle((0.5 - h/2, len(stages)-i-1), h, 0.85,
                          color=color, alpha=0.85)
    ax.add_patch(rect)
    ax.text(0.5, len(stages)-i-0.57, f"{stage}\n{count:,}  ({pct:.1f}%)",
            ha="center", va="center", fontsize=11,
            color="white", fontweight="bold")

ax.set_xlim(0, 1); ax.set_ylim(0, len(stages))
ax.axis("off")
ax.set_title("Order Funnel Shape", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 2. Funnel by City, Cuisine & Payment Mode

In [ ]:

def funnel_df(col):
    grp = full.groupby(col)["order_status"]
    df = pd.DataFrame({
        "Placed"    : grp.count(),
        "Delivered" : grp.apply(lambda x: (x=="Delivered").sum()),
        "Cancelled" : grp.apply(lambda x: (x=="Cancelled").sum()),
        "Refunded"  : grp.apply(lambda x: (x=="Refunded").sum()),
    }).reset_index()
    df["Conversion %"] = df["Delivered"] / df["Placed"] * 100
    df["Cancel %"]     = df["Cancelled"] / df["Placed"] * 100
    df["Refund %"]     = df["Refunded"]  / df["Placed"] * 100
    return df.sort_values("Conversion %", ascending=False)

for col in ["City","cuisine","payment_mode"]:
    df = funnel_df(col)
    display(df.round(2))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, color in zip(axes,
        ["Conversion %","Cancel %","Refund %"],
        ["#2A9D8F","#E63946","#9C6B98"]):
        df_s = df.sort_values(metric)
        ax.barh(df_s[col].astype(str), df_s[metric], color=color)
        ax.set_title(f"{metric} by {col}", fontweight="bold")
        ax.set_xlabel(metric)
        for i, v in enumerate(df_s[metric]):
            ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
    plt.suptitle(f"Funnel Breakdown by {col}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 3. Acquisition Channel → Order Conversion Funnel

In [ ]:

# How many customers from each channel actually placed an order?
cust_with_orders = orders["customer_id"].unique()
customers["placed_order"] = customers["Customer_id"].isin(cust_with_orders)

acq_funnel = customers.groupby("Acquisition_channel").agg(
    signed_up    = ("Customer_id","count"),
    placed_order = ("placed_order","sum"),
).reset_index()
acq_funnel["placed_order"]  = acq_funnel["placed_order"].astype(int)
acq_funnel["conversion_%"]  = acq_funnel["placed_order"] / acq_funnel["signed_up"] * 100

# delivered orders per channel customer
cust_delivered = delivered["customer_id"].unique()
customers["got_delivery"] = customers["Customer_id"].isin(cust_delivered)
acq_funnel["got_delivery"] = customers.groupby("Acquisition_channel")["got_delivery"].sum().values
acq_funnel["delivery_%"]   = acq_funnel["got_delivery"] / acq_funnel["signed_up"] * 100

display(acq_funnel.sort_values("conversion_%", ascending=False).round(2))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(acq_funnel))
w = 0.28
ax.bar(x - w, acq_funnel["signed_up"],   w, label="Signed Up",      color="#264653")
ax.bar(x,     acq_funnel["placed_order"],w, label="Placed Order",   color="#2A9D8F")
ax.bar(x + w, acq_funnel["got_delivery"],w, label="Got Delivery",   color="#43AA8B")
ax.set_xticks(x)
ax.set_xticklabels(acq_funnel["Acquisition_channel"], rotation=15)
ax.set_title("Acquisition → Order → Delivery Funnel by Channel",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Customers")
ax.legend()
for i, row in acq_funnel.iterrows():
    ax.text(i + w, row["got_delivery"] + 5,
            f"{row['delivery_%']:.0f}%", ha="center", fontsize=9, color="#43AA8B")
plt.tight_layout()
plt.show()


## 4. Cohort Retention Analysis

In [ ]:

# Build cohort: first order month per customer
cohort_df = delivered[["customer_id","order_month","order_id"]].copy()
cohort_df["order_month"] = cohort_df["order_month"].astype(str)

first_order = (cohort_df.groupby("customer_id")["order_month"]
               .min().reset_index()
               .rename(columns={"order_month":"cohort_month"}))

cohort_df = cohort_df.merge(first_order, on="customer_id")

# Period index (months since first order)
from pandas import Period
cohort_df["cohort_month_p"]  = pd.PeriodIndex(cohort_df["cohort_month"],  freq="M")
cohort_df["order_month_p"]   = pd.PeriodIndex(cohort_df["order_month"],   freq="M")
cohort_df["period_index"]    = (cohort_df["order_month_p"]
                                - cohort_df["cohort_month_p"]).apply(lambda x: x.n)

cohort_counts = (cohort_df.groupby(["cohort_month","period_index"])["customer_id"]
                 .nunique().unstack())
cohort_sizes  = cohort_counts[0]
retention     = cohort_counts.divide(cohort_sizes, axis=0).round(3)

# Keep only first 12 periods and top 15 cohorts
retention_plot = retention.iloc[:15, :13]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(retention_plot * 100, annot=True, fmt=".0f",
            cmap="YlGn", linewidths=0.4, ax=ax,
            cbar_kws={"label": "Retention %"},
            vmin=0, vmax=100)
ax.set_title("Customer Cohort Retention Rate (%) — Month 0 to Month 12",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Months Since First Order")
ax.set_ylabel("Cohort (First Order Month)")
plt.tight_layout()
plt.show()

avg_retention = retention_plot.mean().dropna()
print("\nAverage retention by period:")
for p, r in avg_retention.items():
    print(f"  Month {p:2d}: {r*100:.1f}%")


## 5. RFM Segmentation

In [ ]:

snapshot_date = delivered["order_timestamp"].max() + pd.Timedelta(days=1)

rfm = delivered.groupby("customer_id").agg(
    recency   = ("order_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency = ("order_id",        "count"),
    monetary  = ("net_revenue",     "sum"),
).reset_index()

# Score 1–5 (5 = best)
rfm["R"] = pd.qcut(rfm["recency"],   5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"],  5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_Score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["RFM_Total"] = rfm["R"] + rfm["F"] + rfm["M"]

def rfm_segment(row):
    r, f, m = row["R"], row["F"], row["M"]
    if r >= 4 and f >= 4 and m >= 4: return "Champions"
    elif r >= 3 and f >= 3:           return "Loyal Customers"
    elif r >= 4 and f <= 2:           return "New Customers"
    elif r <= 2 and f >= 3:           return "At Risk"
    elif r <= 2 and f <= 2:           return "Lost Customers"
    else:                             return "Potential Loyalists"

rfm["Segment"] = rfm.apply(rfm_segment, axis=1)

seg_summary = rfm.groupby("Segment").agg(
    customers  = ("customer_id","count"),
    avg_R      = ("recency","mean"),
    avg_F      = ("frequency","mean"),
    avg_M      = ("monetary","mean"),
    total_rev  = ("monetary","sum"),
).reset_index().sort_values("total_rev", ascending=False)

display(seg_summary.round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = sns.color_palette("Set2", len(seg_summary))

axes[0].pie(seg_summary["customers"], labels=seg_summary["Segment"],
            autopct="%1.1f%%", colors=palette, startangle=140)
axes[0].set_title("Customers by RFM Segment")

axes[1].bar(seg_summary["Segment"], seg_summary["avg_M"],
            color=palette)
axes[1].set_title("Avg Monetary Value by Segment")
axes[1].set_ylabel("₹")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(seg_summary["Segment"], seg_summary["total_rev"],
            color=palette)
axes[2].set_title("Total Revenue by Segment")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"₹{x/1e6:.1f}M"))
axes[2].tick_params(axis="x", rotation=25)

plt.suptitle("RFM Segmentation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

# RFM scatter: Recency vs Frequency coloured by Segment
fig, ax = plt.subplots(figsize=(11, 7))
for seg, grp in rfm.groupby("Segment"):
    ax.scatter(grp["recency"], grp["frequency"],
               s=grp["monetary"]/200, alpha=0.5, label=seg)
ax.set_xlabel("Recency (days since last order)")
ax.set_ylabel("Frequency (# orders)")
ax.set_title("RFM Scatter — Recency vs Frequency
(bubble size = Monetary value)",
             fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 6. Re-order (Repeat Purchase) Funnel

In [ ]:

order_counts = (delivered.groupby("customer_id")["order_id"]
                .count().reset_index()
                .rename(columns={"order_id":"order_count"}))

stages_rp = [1, 2, 3, 5, 10]
labels_rp  = ["≥1 order","≥2 orders","≥3 orders","≥5 orders","≥10 orders"]
counts_rp  = [(order_counts["order_count"] >= s).sum() for s in stages_rp]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_rp = ["#264653","#2A9D8F","#43AA8B","#E9C46A","#E76F51"]
bars = axes[0].bar(labels_rp, counts_rp, color=colors_rp)
for bar, c in zip(bars, counts_rp):
    pct = c / counts_rp[0] * 100
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 10,
                 f"{c:,}
({pct:.1f}%)", ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Repeat Purchase Funnel", fontweight="bold")
axes[0].set_ylabel("Unique Customers")

# Drop-off at each step
dropoffs = [counts_rp[i-1] - counts_rp[i] for i in range(1, len(counts_rp))]
step_labels = [f"{labels_rp[i-1]} → {labels_rp[i]}" for i in range(1, len(labels_rp))]
axes[1].bar(step_labels, dropoffs, color="#E63946", edgecolor="white")
axes[1].set_title("Drop-off Between Repeat Stages", fontweight="bold")
axes[1].set_ylabel("Customers Lost")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(dropoffs):
    pct = v / counts_rp[i] * 100
    axes[1].text(i, v + 5, f"{v:,}
({pct:.1f}%)", ha="center", fontsize=9)

plt.suptitle("Repeat Purchase Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## ✅ Funnel Analysis Summary
- Only **~59.7%** of placed orders successfully deliver — 40.3% lost to cancellations + refunds.
- **Cohort retention drops sharply after Month 1** (~30–40% 2nd-month retention), indicating weak re-engagement.
- **Champions and Loyal Customers** together represent <20% of customers but drive the majority of revenue.
- **At Risk and Lost** segments are large and should be targeted with win-back campaigns.
- **Repeat purchase funnel** shows steep drop from 1st to 2nd order — the first repeat is the hardest conversion.